In [6]:

import matplotlib.pyplot as plt
import numpy as np
import uproot
from scipy.optimize import curve_fit
from scipy.stats import norm
import matplotlib.patches as patches
import math
import sys
import os
sys.path.append(os.path.dirname("/Users/alejandrosanchez/SBND/Work/Tools/Plot_Style/python/"))
import sbnd_style


#Useful functions for analysis
def gauss(x, A, mu, sigma): 
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))
def gaussian_exponential(x, A, mu, sigma, B, lambd):
    gaussian = A * np.exp(-(x - mu)**2 / (2 * sigma**2))
    exponential = B * np.exp(-lambd * x)
    return gaussian + exponential
def expo(x, B, lambd):
    return B * np.exp(-lambd * x)	
def double_gauss(x, A1, mu1, sigma1, A2, mu2, sigma2):
    return A1 * np.exp(-(x - mu1)**2 / (2 * sigma1**2)) + A2 * np.exp(-(x - mu2)**2 / (2 * sigma2**2))

In [3]:
CalibrationFile = uproot.open('/Users/alejandrosanchez/SBND/Work/Calibration/SERNormalization/OV6/Data/pmt_gain_2.2sigma.root')
#Get the channel list from the keys of the root file
channel_list=[]
for key in CalibrationFile.keys():
    if 'amp_opchannel_' in key:
        number = int(key.split('_')[2].split(';')[0])
        channel_list.append(number)
channel_list=np.array(channel_list)

In [9]:
fPlot=False
fit_gauss_expo=False
fit_gauss=True # just a gauss fit seems to work better
fit_double_gauss = False

MeanList=[]
ChannelList=[]
StdList=[]
Chi2List = []
max_idx = []

OffPMTList = np.array([39, 66, 71, 85, 86, 87, 115, 138, 141, 197, 217, 221, 222, 245, 249, 302])

for channel in channel_list:
	if channel in OffPMTList:
		continue
	SERPeakAmplitude = CalibrationFile['PMTGain']['amp_opchannel_'+str(channel)].values()
	x=np.linspace(0, len(SERPeakAmplitude), len(SERPeakAmplitude))
	start_idx = np.argmin(SERPeakAmplitude[9:13])+9
	max_idx.append(np.argmax(SERPeakAmplitude))
	end_idx = start_idx+10

	if(fit_gauss_expo):
		try:
			parameters, covariance = curve_fit(gaussian_exponential, x[start_idx:end_idx]+0.5, SERPeakAmplitude[start_idx:end_idx], p0=[max(SERPeakAmplitude), 13, 2, 1000, 0.01])
			fitted_gauss_expo = gaussian_exponential(x, *parameters)
			fitted_gauss = gauss(x, parameters[0], parameters[1], parameters[2])
			fitted_expo = expo(x, parameters[3], parameters[4])
		except:
			parameters = [0,0,0,0,0]
			fitted_gauss_expo = gaussian_exponential(x, 0, 0, 0, 0, 0)
			fitted_gauss = gauss(x, 0, 0, 0)

	if(fit_gauss):
		try:
			parameters, covariance = curve_fit(gauss, x[start_idx:end_idx]+ 0.5*(x[2]-x[1]), SERPeakAmplitude[start_idx:end_idx], p0=[max(SERPeakAmplitude	), 12, 5])
			fitted_gauss = gauss(x, parameters[0], parameters[1], parameters[2])
		except:
				parameters = [0,0,0]
				fitted_gauss = gauss(x, 0, 0, 0)
	if(fit_double_gauss):
		try:
			parameters, covariance = curve_fit(double_gauss, x[:]+0.5, SERPeakAmplitude[:], p0=[ max(SERPeakAmplitude), 12, 5, max(SERPeakAmplitude), 5, 2])
			fitted_gauss = double_gauss(x+0.5, parameters[0], parameters[1], parameters[2], parameters[3], parameters[4], parameters[5])
		except:
			parameters = [0,0,0,0,0,0]
			fitted_gauss = double_gauss(x, 0, 0, 0, 0, 0, 0)
	if(fPlot):
		fig, ax = plt.subplots(1, figsize=(11, 8))
		ax.plot(SERPeakAmplitude, drawstyle="steps-post")
		ax.plot(x+0.5, fitted_gauss, label='Fit', lw=3)

		ax.set_xlim(0,60)
		ax.grid()
		ax.minorticks_on()
		ax.tick_params(axis='both', which='major', labelsize=18)
		ax.set_xlabel('Peak Amplitude [ADC]', fontsize=18)
		ax.set_ylabel('Counts', fontsize=18)
		ax.set_title('Peak Amplitude Distribution PMT '+str(channel), fontsize=18)
		ax.legend(fontsize=18)
		ax.set_ylim(10,max(SERPeakAmplitude)*1.1)
		ax.set_xlim(0,80)
		ax.text(.65, .5, r'$\mu$ = ' + str(round(parameters[1],2)) + ' ADC', ha='center', va='center', transform=ax.transAxes,  fontsize=18)
		ax.text(.65, .45,r' $\sigma$ = ' + str(round(parameters[2],2)) + '  ADC', ha='center', va='center', transform=ax.transAxes, fontsize=18)
		plt.close(fig)
	if channel in OffPMTList:
		parameters = [0,0,0]
	ChannelList.append(channel)
	MeanList.append(parameters[1])
	StdList.append(parameters[2])



/var/folders/z1/rps0b1355t19p70fmvx5fdpw0000gp/T/ipykernel_30184/664671925.py:16: RuntimeWarning: divide by zero encountered in divide
  return A * np.exp(-(x - mu)**2 / (2 * sigma**2))
/var/folders/z1/rps0b1355t19p70fmvx5fdpw0000gp/T/ipykernel_30184/664671925.py:16: RuntimeWarning: invalid value encountered in divide
  return A * np.exp(-(x - mu)**2 / (2 * sigma**2))


In [ ]:
#List of well-behaved PMTs obtained from the previous plot. This may change over time
channel_list_good= [6, 7, 9,  10,  11,  12,  13,  15,  16,  17,  36, 37,  40,  41,  60,  61, 63,  64,  69,  70, 84,  85,  88,  89,  90,  91, 93,  94,  95, 114, 116, 117, 119, 139, 142, 144, 145, 146, 147, 148, 149, 162, 163, 164, 165, 166, 167, 168, 169, 172, 173, 192, 193, 194, 195, 216, 219, 224, 225, 227, 240, 241, 242, 243, 244, 246, 247, 250, 251, 270, 271, 272, 273, 274, 275, 294, 297, 299, 300, 301, 303, 304, 305]
#Mask for well-behaved PMTs
mask = np.isin(ChannelList, channel_list_good)

#Plot SPE distribution

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
h, b, f = ax.hist(MeanList[mask], bins=60, histtype='step', label='SPE Amplitude', range=[5,20], lw=2)
parameters, covariance = curve_fit(gauss, b[1:]-0.5*(b[1]-b[0]), h, p0=[max(h), 13, 2])
fitted_gauss = gauss(b, parameters[0], parameters[1], parameters[2])

x = np.linspace(8, 14, 1000)
ax.plot(x, gauss(x, parameters[0], parameters[1], parameters[2]), label='Fit', lw=2)
ax.text(.8, .75, r'$\mu$ = ' + str(round(parameters[1],2)) + ' ADC', ha='center', va='center', transform=ax.transAxes,  fontsize=18)
ax.text(.8, .7,r' $\sigma$ = ' + str(round(parameters[2],2)) + '  ADC', ha='center', va='center', transform=ax.transAxes, fontsize=18)
ax.legend()
ax.set_xlabel('Peak Amplitude [ADC]', fontsize=18)
ax.set_ylabel('# Channels', fontsize=18)